In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

In [2]:
# Set random seeds for reproducibility
torch.manual_seed(42)

In [3]:
device  =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [6]:
df = pd.read_csv('/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [8]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [9]:
#Custom transformation for our vgg16 model
import torchvision.transforms as transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406] ,std=[0.229, 0.224, 0.225])
])

In [21]:
from PIL import Image

class MyCustomDataset(Dataset):
  def __init__(self,features,labels,transform):
    self.features = features
    self.labels =labels
    self.transform = transform

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    # resize to (28 * 28)
    image = self.features[index].reshape(28,28)

    # change datatype to np.unit8
    image = image.astype(np.uint8)

    # change black and white to rgb -> (H,W,C) , (C,H,W)
    image = np.stack([image]*3,axis=-1)

    # convert to PIL Image
    image  = Image.fromarray(image)

    # apply tansfprmation
    image = self.transform(image)

    return image, torch.tensor(self.labels[index], dtype=torch.long)

In [22]:
train_dataset = MyCustomDataset(X_train,y_train,transform=custom_transform)
test_dataset = MyCustomDataset(X_test,y_test,transform=custom_transform)

In [33]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True,pin_memory=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [24]:
# import vgg16 model from torchvision
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

In [25]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [26]:
# freeze the existing layers
for param in vgg16.features.parameters():
  param.requires_grad = False

In [27]:
# Define our own calssifier
vgg16.classifier = nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10)
)

In [28]:
vgg16 = vgg16.to(device)

In [29]:
epoch = 10
learning_rate = 0.0001

In [30]:

# loss function
criterion = nn.CrossEntropyLoss()

# Optimizer function
optimizer = optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)


In [31]:
for i in range(epoch):
  total_loss = 0
  for batch_features,batch_labels in train_loader:

    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    #forward pass
    output = vgg16(batch_features)

    #loss calulate
    loss = criterion(output,batch_labels)

    #backward pass
    optimizer.zero_grad()
    loss.backward()

    #update parameters
    optimizer.step()

    total_loss += loss.item()
  print(f"Epoch {i+1} Loss: {total_loss/len(train_loader)}")

Epoch 1 Loss: 0.36573552534977594
Epoch 2 Loss: 0.21739486761577428
Epoch 3 Loss: 0.16900760889425873
Epoch 4 Loss: 0.1323646451147894
Epoch 5 Loss: 0.1057675573193313
Epoch 6 Loss: 0.08422477598061474
Epoch 7 Loss: 0.06976900127231299
Epoch 8 Loss: 0.052717525486911956
Epoch 9 Loss: 0.04771090042207409
Epoch 10 Loss: 0.04297308812791743


In [34]:
# Calculating accuracy

vgg16.eval()

total = 0
correct = 0
with torch.no_grad():
  for batch_features,batch_labels in test_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    output = vgg16(batch_features)
    _,predicted = torch.max(output,1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")



Accuracy: 0.9219166666666667


In [35]:
total = 0
correct = 0
with torch.no_grad():
  for batch_features,batch_labels in train_loader:
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)
    output = vgg16(batch_features)
    _,predicted = torch.max(output,1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()

print(f"Accuracy: {correct/total}")

Accuracy: 0.994875
